# 01 Load and Inspect
Goal: load both sheets of UCI Online Retail II, record what is wrong with the data, and decide what to do about it. The fixes are applied in 02_cleaning.ipynb.

In [1]:
import pandas as pd

path = "../data/raw/online+retail+ii/online_retail_II.xlsx"
sheets = pd.read_excel(path, sheet_name=None, dtype={"Invoice": str, "StockCode": str})

for name, d in sheets.items():
    print(name, d.shape)

Year 2009-2010 (525461, 8)
Year 2010-2011 (541910, 8)


In [2]:
frames = []
for name, d in sheets.items():
    d["source_sheet"] = name
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
df.columns = ["invoice", "stock_code", "description", "quantity",
              "invoice_date", "price", "customer_id", "country", "source_sheet"]
df["customer_id"] = df["customer_id"].astype("Int64")
df.to_csv("../data/processed/online_retail_raw.csv", index=False)
print(df.shape)

(1067371, 9)


In [3]:
import pandas as pd

df = pd.read_csv("../data/processed/online_retail_raw.csv",
                 parse_dates=["invoice_date"],
                 dtype={"invoice": str, "stock_code": str})

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df["invoice_date"].min(), df["invoice_date"].max())
print(df.groupby("source_sheet")["invoice_date"].agg(["min", "max", "count"]))

(1067371, 9)
invoice                    str
stock_code                 str
description                str
quantity                 int64
invoice_date    datetime64[us]
price                  float64
customer_id            float64
country                    str
source_sheet               str
dtype: object
invoice              0
stock_code           0
description       4382
quantity             0
invoice_date         0
price                0
customer_id     243007
country              0
source_sheet         0
dtype: int64
2009-12-01 07:45:00 2011-12-09 12:50:00
                               min                 max   count
source_sheet                                                  
Year 2009-2010 2009-12-01 07:45:00 2010-12-09 20:01:00  525461
Year 2010-2011 2010-12-01 08:26:00 2011-12-09 12:50:00  541910


In [4]:
df["revenue"] = df["quantity"] * df["price"]

print("Customers:", df["customer_id"].nunique())
print("Invoices:", df["invoice"].nunique())

null_cust = df["customer_id"].isna()
print("Missing customer_id: %.1f%% of rows, %.1f%% of revenue"
      % (100*null_cust.mean(), 100*df.loc[null_cust, "revenue"].sum()/df["revenue"].sum()))

print("Cancellation rows (invoice starts with C):", df["invoice"].str.startswith("C").sum())
print("Negative quantity:", (df["quantity"] < 0).sum())
print("Price <= 0:", (df["price"] <= 0).sum())
print("Exact duplicate rows:", df.duplicated().sum())

Customers: 5942
Invoices: 53628
Missing customer_id: 22.8% of rows, 13.7% of revenue
Cancellation rows (invoice starts with C): 19494
Negative quantity: 22950
Price <= 0: 6207
Exact duplicate rows: 12133


In [5]:
overlap = df[(df["invoice_date"] >= "2010-12-01") & (df["invoice_date"] < "2010-12-10")]
print(overlap.groupby("source_sheet").size())

cols = [c for c in df.columns if c != "source_sheet"]
print("Duplicates within a sheet:", df.duplicated().sum())
print("Duplicates ignoring sheet:", df.duplicated(subset=cols).sum())

source_sheet
Year 2009-2010    22523
Year 2010-2011    22523
dtype: int64
Duplicates within a sheet: 12133
Duplicates ignoring sheet: 34335


In [6]:
neg = df[(df["quantity"] < 0) & ~df["invoice"].str.startswith("C")]
print(len(neg))
print("price == 0:", (neg["price"] == 0).mean())
print("no customer_id:", neg["customer_id"].isna().mean())
print(neg["description"].value_counts().head(10))

3457
price == 0: 1.0
no customer_id: 1.0
description
check                     123
damages                    84
?                          83
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Unsaleable, destroyed.      9
Name: count, dtype: int64


In [7]:
bad = df[df["price"] <= 0]
print("no customer_id:", bad["customer_id"].isna().mean())
print(bad["description"].value_counts().head(10))

no customer_id: 0.9885613017560818
description
check                    162
?                         92
damages                   84
damaged                   81
found                     28
missing                   27
sold as set on dotcom     20
Damaged                   17
adjustment                16
OWL DOORSTOP              15
Name: count, dtype: int64


In [8]:
bad = df[df["price"] <= 0]
print("negative price rows:", (bad["price"] < 0).sum())
print(bad.loc[bad["price"] < 0, ["invoice", "description", "price"]].head())

with_cust = bad[bad["customer_id"].notna()]
print("price<=0 with customer_id:", len(with_cust))
print("customers affected:", with_cust["customer_id"].nunique())
print(with_cust[["invoice", "description", "quantity", "price"]].head(10))


negative price rows: 5
        invoice      description     price
179403  A506401  Adjust bad debt -53594.36
276274  A516228  Adjust bad debt -44031.79
403472  A528059  Adjust bad debt -38925.87
825444  A563186  Adjust bad debt -11062.06
825445  A563187  Adjust bad debt -11062.06
price<=0 with customer_id: 71
customers affected: 51
      invoice                      description  quantity  price
4674   489825               6 RIBBONS EMPIRE          12    0.0
6781   489998              DOOR MAT FAIRY CAKE         2    0.0
16107  490727                           Manual         1    0.0
18738  490961   CHRISTMAS PUDDING TRINKET POT          1    0.0
18739  490961     CHRISTMAS CRAFT WHITE FAIRY         12    0.0
32916  492079        ANTIQUE LILY FAIRY LIGHTS         8    0.0
40101  492760  ANTIQUE GLASS HEART DECORATION         12    0.0
47126  493761                  FLAMINGO LIGHTS        24    0.0
48342  493899      CHARLOTTE BAG , SUKI DESIGN        10    0.0
57619  494607        RETRO

In [9]:
# invoice formats other than plain digits or C+digits
odd = df[~df["invoice"].str.match(r"^C?\d{6}$")]
print(odd["invoice"].str[0].value_counts())

# non-product stock codes (not starting with 5 digits)
nonprod = df[~df["stock_code"].str.match(r"^\d{5}")]
print(nonprod["stock_code"].value_counts().head(20))
print("rows:", len(nonprod), " revenue:", round(nonprod["revenue"].sum()))

invoice
A    6
Name: count, dtype: int64
stock_code
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
gift_0001_10      16
CRUK              16
DCGS0076          15
TEST001           15
DCGS0003          14
Name: count, dtype: int64
rows: 6093  revenue: -95225


In [10]:
# 1. How many of the 71 are zero-price vs negative-price?
with_cust = df[(df["price"] <= 0) & df["customer_id"].notna()]
print("zero price:", (with_cust["price"] == 0).sum(),
      " negative price:", (with_cust["price"] < 0).sum())

# 2. Row-level overlap check (all columns except sheet)
cols = [c for c in df.columns if c not in ("source_sheet", "revenue")]
s1 = df[df["source_sheet"] == "Year 2009-2010"]
s2 = df[df["source_sheet"] == "Year 2010-2011"]
s1_end = s1["invoice_date"].max()
s2_overlap = s2[s2["invoice_date"] <= s1_end]
s1_overlap = s1[s1["invoice_date"] >= s2["invoice_date"].min()]

# compare as sorted multisets of rows
a = s1_overlap[cols].sort_values(cols).reset_index(drop=True)
b = s2_overlap[cols].sort_values(cols).reset_index(drop=True)
print(len(a), len(b), "identical:", a.equals(b))

zero price: 71  negative price: 0
22523 22523 identical: True


In [11]:
nonprod = df[~df["stock_code"].str.match(r"^\d{5}")]
summary = (nonprod.groupby("stock_code")
           .agg(rows=("revenue", "size"), revenue=("revenue", "sum"),
                customers=("customer_id", "nunique"))
           .sort_values("revenue", key=abs, ascending=False))
print(summary.head(20).round(0))

# the one extra "A" invoice
print(df[df["invoice"].str.startswith("A")][["invoice", "stock_code", "description", "quantity", "price", "customer_id"]])

# what are the DCGS / PADS / CRUK codes?
print(df[df["stock_code"].str.match(r"^(DCGS|PADS|CRUK)")]
      .groupby(["stock_code", "description"]).size().head(15))

              rows   revenue  customers
stock_code                             
DOT           1446  322647.0          1
AMAZONFEE       43 -260764.0          0
B                6 -147614.0          0
POST          2122  112341.0        496
M             1421  -82796.0        582
BANK CHARGES   102  -35563.0         22
D              177  -13485.0         55
C2             282   13386.0         45
CRUK            16   -7933.0          1
ADJUST          67    6835.0         53
S              104   -6066.0          0
ADJUST2          3     731.0          3
gift_0001_30    29     610.0          0
gift_0001_20    29     471.0          0
DCGSSGIRL       25     296.0          0
DCGS0076        15     292.0          0
gift_0001_50     8     254.0          0
DCGSSBOY        23     251.0          0
TEST001         15     202.0          5
gift_0001_40     7     166.0          0
        invoice stock_code      description  quantity     price  customer_id
179403  A506401          B  Adjust bad debt

In [14]:
s1_end = df.loc[df["source_sheet"] == "Year 2009-2010", "invoice_date"].max()
df2 = df[~((df["source_sheet"] == "Year 2010-2011") & (df["invoice_date"] <= s1_end))].copy()

In [15]:
cols = [c for c in df2.columns if c not in ("source_sheet", "revenue")]
dups = df2[df2.duplicated(subset=cols, keep=False)]
print("duplicate rows:", df2.duplicated(subset=cols).sum())
print("share with no customer_id:", dups["customer_id"].isna().mean().round(3))
print("share that are cancellations:", dups["invoice"].str.startswith("C").mean().round(3))
print(dups.sort_values(["invoice", "stock_code"]).head(10))

duplicate rows: 11812
share with no customer_id: 0.012
share that are cancellations: 0.005
    invoice stock_code                       description  quantity  \
379  489517      21491   SET OF THREE VINTAGE GIFT WRAPS         1   
391  489517      21491   SET OF THREE VINTAGE GIFT WRAPS         1   
365  489517      21821  GLITTER STAR GARLAND WITH BELLS          1   
386  489517      21821  GLITTER STAR GARLAND WITH BELLS          1   
363  489517      21912          VINTAGE SNAKES & LADDERS         1   
371  489517      21912          VINTAGE SNAKES & LADDERS         1   
394  489517      21912          VINTAGE SNAKES & LADDERS         1   
362  489517      21913    VINTAGE SEASIDE JIGSAW PUZZLES         1   
385  489517      21913    VINTAGE SEASIDE JIGSAW PUZZLES         1   
368  489517      22130  PARTY CONE CHRISTMAS DECORATION          6   

           invoice_date  price  customer_id         country    source_sheet  \
379 2009-12-01 11:34:00   1.95      16329.0  United Kingdom

In [16]:
cols = [c for c in df2.columns if c not in ("source_sheet", "revenue")]
df2 = df2.copy()
df2["revenue"] = df2["quantity"] * df2["price"]
extra = df2[df2.duplicated(subset=cols)]          # the repeated copies only
print("revenue in repeated lines:", round(extra["revenue"].sum()),
      "of", round(df2["revenue"].sum()), "total")
print("invoices affected:", extra["invoice"].nunique(), "of", df2["invoice"].nunique())

revenue in repeated lines: 54228 of 18909762 total
invoices affected: 4387 of 53628


In [19]:
pairs = df[df["invoice"].isin(["581483", "C581484", "541431", "C541433"])]
print(pairs[["invoice", "customer_id", "quantity", "revenue", "invoice_date"]])

         invoice  customer_id  quantity   revenue        invoice_date
587080    541431      12346.0     74215   77183.6 2011-01-18 10:01:00
587085   C541433      12346.0    -74215  -77183.6 2011-01-18 10:17:00
1065882   581483      16446.0     80995  168469.6 2011-12-09 09:15:00
1065883  C581484      16446.0    -80995 -168469.6 2011-12-09 09:27:00
